# mr-Diff Training Analysis

This notebook provides tools for analyzing mr-Diff training runs.

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path

from src.utils.logging import load_training_logs
from src.utils.visualization import (
    plot_loss_curves,
    plot_stage_losses,
    plot_predictions,
    plot_trend_decomposition,
    set_style,
)

set_style()
%matplotlib inline

## 1. Load Training Logs

In [ ]:
# Update this path to your log directory
LOG_DIR = "../../logs/mr_diff_YYYYMMDD_HHMMSS"

logs = load_training_logs(LOG_DIR)
config = logs['config']
metrics_df = logs['metrics']

print(f"Config: {config.get('experiment', {})}")
print(f"Metrics shape: {metrics_df.shape if metrics_df is not None else 'None'}")

In [ ]:
# Display first few rows
if metrics_df is not None:
    display(metrics_df.head())

## 2. Loss Curves

In [ ]:
if metrics_df is not None and 'train_loss' in metrics_df.columns:
    train_losses = metrics_df['train_loss'].dropna().tolist()
    val_losses = metrics_df['val_loss'].dropna().tolist() if 'val_loss' in metrics_df.columns else None
    
    fig = plot_loss_curves(
        train_losses=train_losses,
        val_losses=val_losses,
        title='Training and Validation Loss',
        smoothing=0.6,
    )
    plt.show()

## 3. Per-Stage Losses

In [ ]:
if metrics_df is not None:
    stage_cols = [c for c in metrics_df.columns if 'stage_' in c and 'loss' in c]
    
    if stage_cols:
        stage_losses = {}
        for col in stage_cols:
            try:
                stage_num = int(col.split('stage_')[1].split('_')[0])
                stage_losses[stage_num] = metrics_df[col].dropna().tolist()
            except:
                continue
        
        if stage_losses:
            fig = plot_stage_losses(
                stage_losses=stage_losses,
                title='Per-Stage Loss Breakdown',
            )
            plt.show()

## 4. Learning Rate and Gradients

In [ ]:
if metrics_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Learning rate
    if 'train/learning_rate' in metrics_df.columns:
        lr = metrics_df['train/learning_rate'].dropna()
        axes[0].plot(lr.values)
        axes[0].set_xlabel('Step')
        axes[0].set_ylabel('Learning Rate')
        axes[0].set_title('Learning Rate Schedule')
    
    # Gradient norm
    if 'train/gradient_norm' in metrics_df.columns:
        grad_norm = metrics_df['train/gradient_norm'].dropna()
        axes[1].plot(grad_norm.values)
        axes[1].set_xlabel('Step')
        axes[1].set_ylabel('Gradient Norm')
        axes[1].set_title('Gradient Norm')
    
    plt.tight_layout()
    plt.show()

## 5. Prediction Visualization

In [ ]:
# Load predictions if available
PREDICTIONS_PATH = "../../results/predictions.pt"

if Path(PREDICTIONS_PATH).exists():
    data = torch.load(PREDICTIONS_PATH, map_location='cpu')
    predictions = data['predictions']
    targets = data['targets']
    
    print(f"Predictions shape: {predictions.shape}")
    print(f"Targets shape: {targets.shape}")
else:
    print(f"Predictions file not found at {PREDICTIONS_PATH}")
    print("Run evaluate.py --save-predictions first.")

In [ ]:
# Plot sample predictions
if 'predictions' in dir():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for i, ax in enumerate(axes):
        if i >= len(predictions):
            break
            
        pred = predictions[i].numpy()
        target = targets[i].numpy()
        
        # For multivariate, plot first feature
        if pred.ndim > 1:
            pred = pred[:, 0]
            target = target[:, 0]
        
        ax.plot(target, label='Ground Truth', color='blue')
        ax.plot(pred, label='Prediction', color='red', linestyle='--')
        ax.set_title(f'Sample {i+1}')
        ax.legend()
    
    plt.tight_layout()
    plt.show()

## 6. Compare Multiple Runs

In [ ]:
# List your run directories here
RUN_DIRS = [
    # "../../logs/run1",
    # "../../logs/run2",
]

if RUN_DIRS:
    results = {}
    for run_dir in RUN_DIRS:
        logs = load_training_logs(run_dir)
        metrics = logs.get('metrics')
        if metrics is not None and len(metrics) > 0:
            final = metrics.iloc[-1]
            results[Path(run_dir).name] = {
                'train_loss': final.get('train_loss', np.nan),
                'val_loss': final.get('val_loss', np.nan),
                'val_mae': final.get('val_mae', np.nan),
            }
    
    if results:
        df = pd.DataFrame(results).T
        display(df)

## 7. Trend Decomposition Visualization

In [ ]:
# Visualize trend decomposition on sample data
from src.data.preprocessing import TrendExtraction

# Create synthetic example
t = np.linspace(0, 10, 336)
signal = np.sin(t) + 0.5 * np.sin(5*t) + 0.2 * np.sin(20*t) + 0.1 * np.random.randn(len(t))
signal = torch.tensor(signal, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)  # [1, 336, 1]

# Apply trend extraction
kernel_sizes = [5, 25, 51, 201]
trend_extractor = TrendExtraction(kernel_sizes)
trends, residuals = trend_extractor.get_trends_and_residuals(signal)

# Plot
fig = plot_trend_decomposition(
    original=signal[0].numpy(),
    trends=[t[0].numpy() for t in trends],
    residuals=[r[0].numpy() for r in residuals],
    title='Multi-Resolution Trend Decomposition Example',
)
plt.show()